# Seance 1 : le socle technique

**Developpement Python pour l'apprentissage** - MS IA de confiance

---

## Comment utiliser ce notebook

- Les cellules marquées `# A COMPLETER` sont le parcours **obligatoire**.
- Les cellules marquées `# BONUS` sont pour celles et ceux qui avancent vite. Elles ne sont pas nécessaires pour la suite du module.
- Chaque exercice se termine par une cellule `assert` : il s'agit d'un test automatique. Si le test passe sans rien afficher d'autre que le message de réussite, c'est bon.

## Important : les deux pièges du notebook

1. **L'ordre d'execution n'est pas forcément l'ordre de lecture.** Le numero entre crochets à gauche d'une cellule indique dans quel ordre vous l'avez exécuté.
2. **Une variable "cachée" peut rester en mémoire.** Une variable supprimée du code peut toujour exister tant que vous n'avez pas redémarré le noyau (kernel).

> La regle du cours : **avant de rendre quoi que ce soit, faites "Restart and Run All".** Si le notebook ne passe pas de haut en bas sans intervention, il n'est pas fini.

*La création de ce notebook a été assistée par IA générative.*

---

## 0. Diagnostic

Executez la cellule ci-dessous. Si elle affiche des versions, votre environnement est pret.


In [5]:
import sys
print('Python     :', sys.version.split()[0])
print('Executable :', sys.executable)

import numpy as np
print('numpy      :', np.__version__)

try:
    import pandas as pd
    print('pandas     :', pd.__version__)
except ImportError:
    print('pandas     : ABSENT (necessaire seance 2, pas aujourd hui)')

print()
print('Si vous voyez ces lignes, vous etes pret.')


Python     : 3.14.3
Executable : /Users/arnaudmaury/Library/CloudStorage/OneDrive-CentraleSupelec/Enseignement/Séance 1/cours-python-ml/.venv/bin/python
numpy      : 2.5.3
pandas     : 3.0.6

Si vous voyez ces lignes, vous etes pret.


> **Si cette cellule echoue sur `import numpy`**, le probleme le plus frequent est que Visual Studio Code
> n'utilise pas le bon interpreteur. Faites `Ctrl+Shift+P` (`Cmd+Shift+P` sur Mac), tapez
> `Python: Select Interpreter`, et choisissez celui qui contient `.venv`.


---

# Partie 1 : Python, ce qui compte pour le machine learning

Vous avez deja des bases. On ne reprend donc que ce qui bloque reellement en apprentissage automatique.


## 1.1 Rappel express

Lisez, executez, et posez une question si quelque chose vous surprend.


In [2]:
# Types de base
n = 42                 # int
x = 3.14               # float
nom = 'capteur_03'     # str
actif = True           # bool

# Liste : ordonnee, modifiable, heterogene
mesures = [301.2, 299.8, 302.5, 300.1]

# Dictionnaire : association cle -> valeur
machine = {'id': 'L1234', 'type': 'L', 'usure_min': 118}

print(f"{nom} : {len(mesures)} mesures, moyenne {sum(mesures) / len(mesures):.2f} K")
print(f"machine {machine['id']} de type {machine['type']}")


capteur_03 : 4 mesures, moyenne 300.90 K
machine L1234 de type L


In [3]:
# Boucle et condition
for m in mesures:
    if m > 301:
        print(f'{m} K : au-dessus du seuil')
    else:
        print(f'{m} K : nominal')


301.2 K : au-dessus du seuil
299.8 K : nominal
302.5 K : au-dessus du seuil
300.1 K : nominal


## 1.2 Les compréhensions de liste

Plus courtes, plus lisibles, et plus rapides qu'une boucle avec `append`. Vous en verrez partout.


In [4]:
# Version longue
celsius = []
for k in mesures:
    celsius.append(k - 273.15)

# Version en comprehension : meme resultat, une ligne
celsius2 = [k - 273.15 for k in mesures]

# Avec un filtre
chaudes = [k for k in mesures if k > 301]

print(celsius)
print(celsius2)
print(chaudes)


[28.05000000000001, 26.650000000000034, 29.350000000000023, 26.950000000000045]
[28.05000000000001, 26.650000000000034, 29.350000000000023, 26.950000000000045]
[301.2, 302.5]


In [ ]:
# --- EXERCICE 1 : A COMPLETER ---------------------------------------
# A partir de la liste 'couples' ci-dessous, construisez en UNE comprehension
# la liste 'surcouples' des valeurs strictement superieures a 60.

couples = [38.2, 41.0, 62.5, 29.9, 71.3, 55.0, 60.0, 68.8]

surcouples = ...   # <- remplacez les points de suspension


In [ ]:
assert surcouples == [62.5, 71.3, 68.8], f'Obtenu {surcouples}'
print('Exercice 1 : correct')


## 1.3 Les fonctions, et pourquoi leurs signatures comptent

Les bibliotheques que vous allez utiliser sont pleines d'arguments nommes avec des valeurs par defaut.
Savoir lire une signature, c'est savoir lire la documentation.


In [ ]:
def puissance(couple, vitesse_rpm, rendement=1.0):
    """Puissance mecanique en watts.

    couple      : couple en Nm
    vitesse_rpm : vitesse de rotation en tours par minute
    rendement   : facteur entre 0 et 1 (defaut : 1.0, cas ideal)
    """
    import math
    omega = vitesse_rpm * 2 * math.pi / 60   # rad/s
    return couple * omega * rendement

# Trois facons d'appeler la meme fonction
print(puissance(40, 1500))                             # positionnel
print(puissance(couple=40, vitesse_rpm=1500))          # nomme, plus lisible
print(puissance(40, 1500, rendement=0.92))             # on change le defaut

# La docstring est lisible depuis le notebook :
help(puissance)


## 1.4 Les objets : pourquoi on ecrit `model.fit(X, y)`

C'est le point le plus important de cette partie. En seance 3 vous ecrirez :

```python
model = LogisticRegression(C=1.0)   # on CONSTRUIT un objet, avec ses reglages
model.fit(X, y)                     # on lui demande d'apprendre
model.predict(X_test)               # on lui demande de predire
model.coef_                         # on lit ce qu'il a appris
```

Et en seance 4, avec PyTorch, exactement la meme logique sous d'autres noms.
Construisons donc un petit objet nous-memes pour que le mecanisme soit clair.


In [ ]:
class Capteur:
    def __init__(self, nom, seuil=301.0):
        # Les REGLAGES, choisis par vous a la construction.
        self.nom = nom
        self.seuil = seuil
        # Ce qui sera APPRIS des donnees. Convention scikit-learn :
        # un attribut qui se termine par _ n'existe qu'apres l'apprentissage.
        self.moyenne_ = None

    def fit(self, mesures):
        """Apprend a partir des donnees."""
        self.moyenne_ = sum(mesures) / len(mesures)
        return self          # scikit-learn renvoie toujours self

    def predict(self, mesures):
        """Applique ce qui a ete appris."""
        if self.moyenne_ is None:
            raise RuntimeError('Appelez fit() avant predict().')
        return [m > self.seuil for m in mesures]


c = Capteur('temperature_air', seuil=300.5)
c.fit(mesures)
print('moyenne apprise :', round(c.moyenne_, 2))
print('alertes         :', c.predict(mesures))


**Retenez la distinction**, elle structure tout le reste du mastere :

| | Ou | Qui le choisit | Exemple |
|---|---|---|---|
| **Hyperparametre** | dans le constructeur | vous | `seuil=300.5`, `C=1.0` |
| **Parametre appris** | attribut finissant par `_` | les donnees | `moyenne_`, `coef_` |


## 1.5 Lire une erreur

Un message d'erreur Python se lit **du bas vers le haut** : la derniere ligne dit ce qui ne va pas,
les lignes au-dessus disent ou.

Executez la cellule suivante, qui echoue volontairement, et lisez le message.


In [ ]:
donnees = {'couple': 40.2, 'vitesse': 1500}
print(donnees['torque'])   # echoue : la cle s'appelle 'couple', pas 'torque'


Les cinq erreurs que vous rencontrerez le plus :

| Erreur | Ce qu'elle veut dire | Premier reflexe |
|---|---|---|
| `ModuleNotFoundError` | la bibliotheque n'est pas installee **dans cet interpreteur** | verifier l'interpreteur selectionne, puis `pip install` |
| `NameError` | la variable n'existe pas encore | la cellule qui la cree n'a pas ete executee |
| `KeyError` | cette cle n'existe pas dans le dictionnaire | afficher les cles disponibles |
| `ValueError: shapes not aligned` | deux tableaux de formes incompatibles | afficher les `.shape` |
| `IndentationError` | melange d'espaces et de tabulations | laisser l'editeur reformater |


---

# Partie 2 : numpy, penser en tableaux

C'est la partie la plus rentable de la journee. Tout le deep learning est du numpy deguise,
et les erreurs de **forme** sont ce qui vous bloquera le plus dans les cours suivants.


## 2.1 Pourquoi numpy : la demonstration


In [ ]:
import numpy as np

n = 1_000_000
liste = list(range(n))
tableau = np.arange(n)

print('Boucle Python :')
%timeit sum(x * x for x in liste)

print('numpy vectorise :')
%timeit (tableau * tableau).sum()


Le rapport est d'un facteur 50 a 100. La raison : numpy execute la boucle en C sur un bloc
de memoire contigu, au lieu de manipuler un million d'objets Python.

> **Consequence pratique :** des que vous ecrivez une boucle `for` sur des donnees numeriques,
> demandez-vous s'il existe une ecriture vectorisee. En general, oui.


## 2.2 La forme est reine


In [ ]:
a = np.array([1, 2, 3, 4])                    # 1 dimension
b = np.array([[1, 2, 3], [4, 5, 6]])          # 2 dimensions

for nom, t in [('a', a), ('b', b)]:
    print(f'{nom} : shape={t.shape}  ndim={t.ndim}  dtype={t.dtype}')


> **Le reflexe a installer aujourd'hui : en cas de doute, affichez `.shape`.**
> C'est la reponse a la grande majorite des erreurs que vous rencontrerez en seance 4
> et dans les cours de deep learning.


In [ ]:
# Creation
print(np.zeros((2, 3)))
print(np.ones(4))
print(np.arange(0, 10, 2))          # de 0 a 10 exclu, par pas de 2
print(np.linspace(0, 1, 5))         # 5 valeurs reparties de 0 a 1 inclus

# Aleatoire, avec une graine : indispensable pour que vos resultats soient reproductibles
rng = np.random.default_rng(42)
print(rng.normal(loc=40, scale=10, size=5).round(2))


> **Reproductibilite.** La graine (`42` ici) fixe la suite aleatoire. Sans elle, deux executions
> de votre notebook donnent des resultats differents, et personne ne peut verifier votre travail.
> Dans un contexte d'IA de confiance, un modele non reproductible n'est pas auditable.


## 2.3 Indexation, decoupage, masques booleens


In [ ]:
rng = np.random.default_rng(0)
couple = rng.normal(40, 10, 12).round(1)
panne  = (couple > 55).astype(int)

print('couple :', couple)
print('panne  :', panne)
print()
print('premier element     :', couple[0])
print('dernier element     :', couple[-1])
print('trois premiers      :', couple[:3])
print('un sur deux         :', couple[::2])
print()
# Le masque booleen : LA brique que vous utiliserez partout
print('masque              :', couple > 55)
print('valeurs > 55        :', couple[couple > 55])
print('couples en panne    :', couple[panne == 1])
print('combien de pannes   :', (panne == 1).sum())


## 2.4 Agregations et l'argument `axis`

C'est le piege classique. Sur un tableau a deux dimensions de forme `(lignes, colonnes)` :

```
axis=0  ->  on ecrase les LIGNES, on obtient un resultat PAR COLONNE
axis=1  ->  on ecrase les COLONNES, on obtient un resultat PAR LIGNE
```

Moyen mnemotechnique : `axis=k` est l'axe qui **disparait** du resultat.


In [ ]:
X = np.array([[300.1, 40.2, 1500],
              [301.5, 38.9, 1480],
              [299.8, 44.1, 1550],
              [302.2, 39.5, 1520]])

print('X.shape            :', X.shape)             # (4 lignes, 3 colonnes)
print('X.mean()           :', X.mean().round(2))   # tout
print('X.mean(axis=0)     :', X.mean(axis=0).round(2), ' <- une valeur par COLONNE')
print('X.mean(axis=1)     :', X.mean(axis=1).round(2), ' <- une valeur par LIGNE')
print()
print('shape axis=0 :', X.mean(axis=0).shape, ' | shape axis=1 :', X.mean(axis=1).shape)


## 2.5 Le broadcasting

numpy sait combiner deux tableaux de formes differentes en etirant automatiquement le plus petit.

La regle, en comparant les formes **de droite a gauche** : deux dimensions sont compatibles si elles
sont egales, ou si l'une vaut 1.

```
X       (4, 3)
moyenne    (3,)     ->  etiree en (4, 3)   OK

X       (4, 3)
colonne (4, 1)      ->  etiree en (4, 3)   OK

X       (4, 3)
autre      (4,)     ->  3 et 4 incompatibles   ERREUR
```


In [ ]:
moyennes = X.mean(axis=0)          # shape (3,)
print('X        :', X.shape)
print('moyennes :', moyennes.shape)
print()
print('X - moyennes (centrage colonne par colonne) :')
print((X - moyennes).round(2))


In [ ]:
# Le cas qui echoue, et le message qu'il produit
par_ligne = X.mean(axis=1)          # shape (4,)
try:
    X - par_ligne
except ValueError as e:
    print('ValueError :', e)

# La correction : rendre la forme explicite avec reshape
print()
print('Corrige avec par_ligne.reshape(-1, 1), de forme', par_ligne.reshape(-1, 1).shape, ':')
print((X - par_ligne.reshape(-1, 1)).round(2))


---

# TP : trois exercices

Chaque exercice a sa cellule de verification juste en dessous.


## Exercice 2 : standardisation (z-score)

Standardiser une matrice, c'est ramener **chaque colonne** a une moyenne de 0 et un ecart-type de 1 :

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

C'est exactement ce que fait le `StandardScaler` de scikit-learn que vous verrez en seance 3.

**Contrainte : aucune boucle `for`.**


In [ ]:
# --- EXERCICE 2 : A COMPLETER ---------------------------------------
def standardiser(M):
    """Centre et reduit chaque colonne de M. M est de forme (n_lignes, n_colonnes)."""
    # Indice : M.mean(axis=?) et M.std(axis=?), puis le broadcasting fait le reste.
    ...


Z = standardiser(X)
print(Z.round(3))


In [ ]:
assert Z.shape == X.shape, f'Forme attendue {X.shape}, obtenue {Z.shape}'
assert np.allclose(Z.mean(axis=0), 0, atol=1e-9), 'Les moyennes par colonne ne sont pas nulles'
assert np.allclose(Z.std(axis=0), 1, atol=1e-9), 'Les ecarts-types par colonne ne valent pas 1'
print('Exercice 2 : correct')


## Exercice 3 : distances sans boucle

Calculez la distance euclidienne entre un point de reference et **chacune** des lignes d'une matrice.

$$d_i = \sqrt{\sum_j (x_{ij} - p_j)^2}$$

**Contrainte : aucune boucle `for`.**

> Ce que vous venez d'ecrire est le coeur de l'algorithme des k plus proches voisins.
> Vous le retrouverez en seance 3 sous le nom de `KNeighborsClassifier`.


In [ ]:
# --- EXERCICE 3 : A COMPLETER ---------------------------------------
rng = np.random.default_rng(7)
points = rng.normal(0, 1, size=(1000, 3))    # 1000 points en dimension 3
reference = np.array([0.5, -0.2, 1.0])

def distances(points, reference):
    """Renvoie un tableau de forme (n_points,) : la distance de chaque point a la reference."""
    # Indice : (points - reference) marche par broadcasting.
    # Ensuite : elever au carre, sommer sur le bon axe, prendre la racine (np.sqrt).
    ...


d = distances(points, reference)
print('forme  :', d.shape)
print('min    :', d.min().round(3))
print('plus proche voisin : ligne', d.argmin())


In [ ]:
assert d.shape == (1000,), f'Forme attendue (1000,), obtenue {d.shape}'
attendu = np.sqrt(((points - reference) ** 2).sum(axis=1))
assert np.allclose(d, attendu), 'Les distances ne sont pas les bonnes'
print('Exercice 3 : correct')


## Exercice 4 (BONUS) : regression lineaire par moindres carres

Ajustez une droite $y = a x + b$ sur des donnees bruitees, de deux facons :

1. avec `np.linalg.lstsq`, qui resout le probleme des moindres carres,
2. avec la formule fermee $\hat{\beta} = (X^T X)^{-1} X^T y$.

Les deux doivent donner le meme resultat. C'est votre premier modele d'apprentissage,
et il tient en trois lignes.


In [ ]:
# --- EXERCICE 4 (BONUS) : A COMPLETER -------------------------------
rng = np.random.default_rng(1)
x = np.linspace(0, 10, 50)
y = 2.5 * x + 1.0 + rng.normal(0, 1.5, 50)   # vraie pente 2.5, vraie ordonnee 1.0

# 1. Construisez la matrice de design A, de forme (50, 2) :
#    la premiere colonne est x, la seconde ne contient que des 1 (pour l'ordonnee a l'origine).
#    Indice : np.column_stack, et np.ones_like(x)
A = ...

# 2. Resolvez avec np.linalg.lstsq(A, y, rcond=None) ; le resultat est un tuple,
#    les coefficients sont le premier element.
coef_lstsq = ...

# 3. Refaites le calcul avec la formule fermee.
#    Indice : A.T @ A, np.linalg.inv, et A.T @ y
coef_formule = ...

print('lstsq   : pente %.3f, ordonnee %.3f' % (coef_lstsq[0], coef_lstsq[1]))
print('formule : pente %.3f, ordonnee %.3f' % (coef_formule[0], coef_formule[1]))


In [ ]:
assert A.shape == (50, 2), f'A doit etre de forme (50, 2), obtenu {A.shape}'
assert np.allclose(coef_lstsq, coef_formule), 'Les deux methodes doivent coincider'
assert abs(coef_lstsq[0] - 2.5) < 0.3, 'La pente estimee est loin de 2.5'
print('Exercice 4 : correct')


---

# Ce qu'il faut retenir de la seance 1

1. **Un environnement isole et des versions figees**, parce qu'un modele qu'on ne peut pas
   re-executer a l'identique n'est pas auditable.
2. **`model.fit(X, y)`** : hyperparametres dans le constructeur, parametres appris dans les
   attributs finissant par `_`. Meme logique dans scikit-learn et dans PyTorch.
3. **La forme est reine.** En cas de doute, affichez `.shape`.
4. **`axis=k` est l'axe qui disparait.**
5. **Une boucle `for` sur des donnees numeriques est presque toujours remplacable.**

## Pour la seance 2 (30 minutes)

Reprenez les exercices 2 et 3 sans regarder la solution. Si vous les refaites sans hesiter,
vous avez le socle numpy dont vous aurez besoin toute l'annee.
